# MEW public-evidence quality audit
Run from the repository root or research/learning. Standard library only. Hash checks prove retained bytes, not source truth or training rights. Fees are exact integer lovelace; no customer payments or correlations inferred.


In [1]:
from pathlib import Path
import json, hashlib
root = Path.cwd()
if not (root / "research/learning").exists():
    root = root.parent.parent
snapshot = root / "research/learning/2026-10-07T13-13-06-688Z"
quality = json.loads((snapshot / "quality.json").read_text())
raw_records = (snapshot / "records.jsonl").read_bytes()
assert hashlib.sha256(raw_records).hexdigest() == quality["snapshotSha256"]
records = [json.loads(line) for line in raw_records.splitlines()]
assert len({row["id"] for row in records}) == len(records)
manifest = json.loads((snapshot / "manifest.json").read_text())
verified = 0
for source in manifest:
    if source["status"] == "collected":
        raw = (snapshot / source["rawFile"]).read_bytes()
        assert hashlib.sha256(raw).hexdigest() == source["sha256"]
        verified += 1
assert all(row["trainingEligible"] is False and row["label"] is None for row in records)
counts = {grain: sum(row["grain"] == grain for row in records) for grain in sorted({row["grain"] for row in records})}
for entity in {row["entity"] for row in records if row["grain"] == "article_day"}:
    days = sorted(row["eventAt"] for row in records if row["grain"] == "article_day" and row["entity"] == entity)
    assert len(days) == len(set(days)) == 28
    from datetime import datetime, timedelta
    times = [datetime.fromisoformat(day.replace("Z", "+00:00")) for day in days]
    assert all(b-a == timedelta(days=1) for a,b in zip(times, times[1:]))
fees = {network: str(sum(int(row["feeLovelace"]) for row in records if row["grain"] == "network_transaction" and row["network"] == network)) for network in ["mainnet", "preprod"]}
print(json.dumps({"records":len(records),"verifiedRawSources":verified,"grains":counts,"feeLovelaceSum":fees,"fineTuningReady":False}, indent=2))


{
  "records": 159,
  "verifiedRawSources": 15,
  "grains": {
    "article_day": 112,
    "network_block": 10,
    "network_transaction": 32,
    "repository_snapshot": 5
  },
  "feeLovelaceSum": {
    "mainnet": "4476085",
    "preprod": "4140620"
  },
  "fineTuningReady": false
}
